# Analyzing an Author's Corpus with RAG

This notebook is the interactive front end for source-agnostic ingestion, exact catalog queries, and semantic retrieval. Reusable implementation lives in the `author_corpus` package.

## Local configuration and privacy

Copy `corpus.example.yaml` to the ignored `corpus.local.yaml`, configure private input paths there, and set `AUTHOR_CORPUS_CONFIG`. Select the `Python 3.14 (author-corpus-rag)` kernel before running the notebook. Committed notebook outputs are stripped automatically.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

import author_corpus
from author_corpus import (
    INDEX_PIPELINE_VERSION,
    CacheLayout,
    CorpusCatalog,
    NotebookRunOptions,
    TimingLog,
    build_catalog_tools,
    corpus_fingerprint,
    load_corpus_config,
)
from author_corpus.persistence import write_manifest

timings = TimingLog()
project_root = Path(author_corpus.__file__).resolve().parents[2]
load_dotenv(project_root / ".env")
run_options = NotebookRunOptions.from_environment(os.environ)
print("allow missing index build:", run_options.allow_index_build)
print("run grounded answer:", run_options.run_grounded_answer)

config_value = os.getenv("AUTHOR_CORPUS_CONFIG")
if not config_value:
    raise RuntimeError("Set AUTHOR_CORPUS_CONFIG to an ignored corpus.local.yaml file.")

config_path = Path(config_value).expanduser()
if not config_path.is_absolute():
    config_path = project_root / config_path
config_path = config_path.resolve()

## Load and validate logical documents

Missing metadata is reported explicitly but does not prevent semantic indexing. Blocking extraction problems fail before indexes are built.

In [ ]:
load_started = timings.start()
load_result = load_corpus_config(config_path)

print("corpus:", load_result.name or config_path.stem)
print("documents:", len(load_result.documents))
print("warnings:", len(load_result.warnings))
print("errors:", len(load_result.errors))

for issue in load_result.issues:
    print(f"{issue.severity.upper():7} {issue.code}: {issue.path or '<corpus>'} — {issue.message}")

load_result.raise_for_errors()
documents = load_result.documents
print(timings.finish("Load and validate corpus", load_started))

## Build the exact catalog

Counts and exhaustive metadata questions belong here, not in top-k vector retrieval.

In [ ]:
catalog_started = timings.start()
index_options = {
    "chunk_size": 1024,
    "chunk_overlap": 200,
    "index_pipeline_version": INDEX_PIPELINE_VERSION,
    "embedding_model": os.getenv("EMBEDDING_MODEL", "BAAI/bge-small-en-v1.5"),
}
fingerprint = corpus_fingerprint(documents, options=index_options)
layout = CacheLayout(project_root / ".cache", fingerprint)
layout.ensure()

catalog = CorpusCatalog(layout.catalog_path)
catalog.rebuild(documents)
write_manifest(layout, documents, options=index_options)

catalog_stats = catalog.stats()
print(timings.finish("Build exact catalog", catalog_started))
catalog_stats

In [ ]:
catalog.author_counts()

In [ ]:
catalog_tools = build_catalog_tools(catalog)
[tool.metadata.name for tool in catalog_tools]

## Build or load retrieval profiles

The vector and BM25 indexes are stored under the corpus fingerprint, so changed documents or chunking settings produce new cache entries. Loading existing indexes is safe during `Run All`; building a missing vector index requires `AUTHOR_CORPUS_ALLOW_INDEX_BUILD=1`.

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from author_corpus.indexing import load_or_build_vector_index, load_vector_index, vector_index_exists

vector_started = timings.start()
embed_model = HuggingFaceEmbedding(
    model_name=index_options["embedding_model"],
    device="cpu",
)
if vector_index_exists(layout):
    vector_index = load_vector_index(layout, embed_model=embed_model)
    loaded_from_cache = True
elif run_options.allow_index_build:
    vector_index, loaded_from_cache = load_or_build_vector_index(
        documents,
        layout,
        embed_model=embed_model,
        chunk_size=index_options["chunk_size"],
        chunk_overlap=index_options["chunk_overlap"],
    )
else:
    raise RuntimeError(
        "The vector index is missing. Set AUTHOR_CORPUS_ALLOW_INDEX_BUILD=1, restart the kernel, and rerun this cell."
    )
print("loaded from cache:", loaded_from_cache)
print(timings.finish("Load or build vector index", vector_started))

## Inspect semantic evidence

Retrieval is intentionally non-exhaustive. Broad discovery uses dense retrieval with one passage per document. Focused answers use dense plus BM25 reciprocal-rank fusion and may retain three passages from one document. Each result exposes the score type, component ranks, source URI, and exact passage before an LLM is allowed to synthesize an answer.

In [ ]:
from author_corpus import build_retrieval_profiles

profiles_started = timings.start()
retrieval_profiles, lexical_loaded_from_cache = build_retrieval_profiles(
    vector_index,
    layout,
    candidate_pool_size=30,
    evidence_passages_per_document=3,
)
discovery_search = retrieval_profiles.discovery
evidence_search = retrieval_profiles.hybrid_evidence
print("lexical index loaded from cache:", lexical_loaded_from_cache)
print(timings.finish("Load or build retrieval profiles", profiles_started))

In [ ]:
semantic_query = "What practical advice does the author give readers?"
search_started = timings.start()
minimum_document_author_fraction = 0.8
semantic_result = discovery_search.search(
    semantic_query,
    minimum_document_author_fraction=minimum_document_author_fraction,
)
search_timing = timings.finish("Semantic retrieval", search_started)

print("query:", semantic_result.query)
print("strategy:", semantic_result.strategy)
print("score kind:", semantic_result.score_kind)
print("exhaustive:", semantic_result.exhaustive)
print("candidates inspected:", semantic_result.inspected_candidates)

for passage in semantic_result.passages:
    score = "n/a" if passage.score is None else f"{passage.score:.4f}"
    print(f"\n{passage.rank}. {passage.title} — score {score}")
    contributions = ", ".join(
        f"{item.method} rank {item.rank} ({item.score_kind}={item.score:.4f})"
        for item in passage.retrieval_contributions
        if item.score is not None
    )
    print("   retrieval:", contributions or "<not recorded>")
    print("   document:", passage.document_id)
    print("   source:", passage.canonical_source_uri or "<no canonical source>")
    print("   section:", " > ".join(passage.section_path) or "<document introduction>")
    author_fraction = passage.document_author_fraction
    author_percent = "unknown" if author_fraction is None else f"{author_fraction:.1%}"
    quoted_fraction = passage.quoted_speech_fraction
    quoted_percent = "unknown" if quoted_fraction is None else f"{quoted_fraction:.1%}"
    uncertain_fraction = passage.uncertain_voice_fraction
    uncertain_percent = "unknown" if uncertain_fraction is None else f"{uncertain_fraction:.1%}"
    print(
        f"   voice: {passage.passage_voice}; author: {author_percent}; quoted: {quoted_percent}; uncertain: {uncertain_percent}"
    )
    if passage.attributed_speakers:
        print("   attributed speakers:", ", ".join(passage.attributed_speakers))
    print("   passage:", passage.text[:500].replace("\n", " "))

print(f"\n{search_timing}")

## Generate a grounded answer

The model receives only the retrieved passages, must use numbered evidence markers, and returns the cited sources as clickable links. Generation is disabled during `Run All` unless `AUTHOR_CORPUS_RUN_GROUNDED_ANSWER=1`. Continue to use the exact catalog—not this semantic answerer—for totals and exhaustive lists.

In [ ]:
from IPython.display import Markdown, display

from author_corpus import GroundedAnswerer
from author_corpus.local_llm import LocalModelSettings, OpenAICompatibleCompleter

local_model_settings = None
grounded_answerer = None
if run_options.run_grounded_answer:
    model_id = os.getenv("OLLAMA_MODEL")
    if not model_id or model_id == "your-local-chat-model":
        raise RuntimeError("Set OLLAMA_MODEL to an installed local chat model.")
    local_model_settings = LocalModelSettings(
        model_id=model_id,
        base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
        reasoning_effort=os.getenv("OLLAMA_REASONING_EFFORT", "none"),
    )
    grounded_answerer = GroundedAnswerer(
        evidence_search,
        OpenAICompatibleCompleter(local_model_settings),
        model_id=local_model_settings.model_id,
        evidence_limit=6,
    )
else:
    print("Grounded generation skipped; semantic evidence above is still available for inspection.")

In [ ]:
from author_corpus import GenerationTraceSettings, QueryTrace, QueryTraceStore, RetrievalTraceSettings

grounded_answer = None
query_trace = None
if grounded_answerer is None or local_model_settings is None:
    print("No model call or query trace was created.")
else:
    answer_started = timings.start()
    grounded_answer = grounded_answerer.answer(
        semantic_query,
        minimum_document_author_fraction=minimum_document_author_fraction,
    )
    answer_timing = timings.finish("Grounded answer generation", answer_started)
    display(Markdown(grounded_answer.to_markdown()))
    print(answer_timing)

    trace_started = timings.start()
    query_trace = QueryTrace.from_grounded_answer(
        grounded_answer,
        corpus_fingerprint=fingerprint,
        document_content_hashes={document.document_id: document.content_hash for document in documents},
        retrieval=RetrievalTraceSettings(
            evidence_limit=grounded_answerer.evidence_limit,
            max_passages_per_document=evidence_search.max_passages_per_document,
            minimum_document_author_fraction=minimum_document_author_fraction,
            strategy=evidence_search.strategy,
            score_kind=evidence_search.score_kind,
        ),
        generation=GenerationTraceSettings.from_local_settings(
            local_model_settings,
            prompt_version=grounded_answer.prompt_version,
        ),
        elapsed_seconds=answer_timing.elapsed_seconds,
    )
    QueryTraceStore(layout.query_trace_path).put(query_trace)
    print("query trace:", query_trace.trace_id)
    print(timings.finish("Persist query trace", trace_started))

## Run the local retrieval evaluation

Corpus-specific queries and exact passage judgments live in an ignored YAML file. Dense, lexical, and hybrid strategies are timed and measured on the same cases before a default changes. Document- and passage-level hit rate, MRR, and nDCG are reported separately.

In [ ]:
from author_corpus import evaluate_retrieval_strategies, load_retrieval_cases

evaluation_value = os.getenv("AUTHOR_CORPUS_RETRIEVAL_EVAL")
if not evaluation_value:
    print("Set AUTHOR_CORPUS_RETRIEVAL_EVAL to run local retrieval cases.")
else:
    evaluation_path = Path(evaluation_value).expanduser()
    if not evaluation_path.is_absolute():
        evaluation_path = project_root / evaluation_path
    evaluation_cases = load_retrieval_cases(evaluation_path.resolve())
    evaluation_started = timings.start()
    benchmark = evaluate_retrieval_strategies(
        retrieval_profiles.evaluation_searches(),
        evaluation_cases,
        top_k=6,
    )
    print("cases:", len(evaluation_cases))
    for strategy_result in benchmark.strategies:
        evaluation = strategy_result.evaluation
        document_metrics = (
            f"document hit={evaluation.hit_rate:.1%}, MRR={evaluation.mean_reciprocal_rank:.3f}, "
            f"nDCG={evaluation.mean_normalized_discounted_cumulative_gain:.3f}"
        )
        if evaluation.passage_hit_rate is None:
            passage_metrics = "passage metrics=<no passage labels>"
        else:
            passage_metrics = (
                f"passage hit={evaluation.passage_hit_rate:.1%}, "
                f"MRR={evaluation.passage_mean_reciprocal_rank:.3f}, "
                f"nDCG={evaluation.passage_mean_normalized_discounted_cumulative_gain:.3f}"
            )
        print(
            f"{strategy_result.strategy}: {document_metrics}; {passage_metrics}; elapsed={strategy_result.elapsed_seconds:.3f}s"
        )
    hybrid_evaluation = next(item.evaluation for item in benchmark.strategies if item.strategy == "hybrid_rrf")
    for case in hybrid_evaluation.cases:
        message = f"{case.name}: first document rank = {case.first_relevant_rank}"
        if case.has_passage_expectations:
            message += f"; first passage rank = {case.first_relevant_passage_rank}"
        print(message)
    print(timings.finish("Retrieval evaluation", evaluation_started))

## Inspect cached navigation summaries

This notebook only reports cache coverage. It never resumes summary generation or corpus-wide synthesis during `Run All`. Build resumable per-document navigation summaries with `uv run --extra local author-corpus build-knowledge`. The unaudited broad synthesis remains available only through the explicit `--include-experimental-synthesis` flag.

In [ ]:
from author_corpus import SummaryStore
from author_corpus.summaries import DOCUMENT_SUMMARY_PROMPT_VERSION

knowledge_store = SummaryStore(layout.summary_path)
knowledge_model_id = os.getenv("OLLAMA_MODEL", "").strip()
if not knowledge_model_id or knowledge_model_id == "your-local-chat-model":
    print("Set OLLAMA_MODEL to inspect model-specific navigation-summary coverage.")
else:
    status_started = timings.start()
    missing_summaries = knowledge_store.missing(
        documents,
        model_id=knowledge_model_id,
        prompt_version=DOCUMENT_SUMMARY_PROMPT_VERSION,
    )
    print("cached navigation summaries:", len(documents) - len(missing_summaries))
    print("missing navigation summaries:", len(missing_summaries))
    print("broad synthesis: paused pending claim-level evidence auditing")
    print(timings.finish("Inspect navigation-summary cache", status_started))

In [ ]:
print("No corpus synthesis is generated or displayed by this notebook.")

## Timing report

Compare cold-build and warm-cache timings before changing models, chunking, or persistence behavior.

In [ ]:
print(timings.format_report())

## Next milestones

- Extract answer claims into exact, versioned source spans.
- Classify claims as supported, qualified, contradicted, or unsupported.
- Evaluate known corrections, disagreements, quotations, and changing facts.
- Resume broad synthesis only after its claims can be audited against raw text.
- Route broad discovery and focused evidence queries automatically only after more labeled cases justify the router.
- Add conversational memory and a small Gradio interface after the audit milestone.